# Supplementary Figure S4 — species phylogeny

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
from io import StringIO
import re
import matplotlib.pyplot as plt
from Bio import Phylo

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/phylogeny/Gray_et_al_Gryllus_best_species_tree.nex").is_file())
SOURCE = ROOT / "data/phylogeny/Gray_et_al_Gryllus_best_species_tree.nex"
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

raw = SOURCE.read_text()
taxa = re.search(r"TAXLABELS\s*(.*?)\s*;", raw, re.I | re.S).group(1).split()
newick = re.search(r"TREE\s+\*\s+UNTITLED\s*=\s*\[&R\]\s*(.*?);", raw, re.I | re.S).group(1) + ";"
tree = Phylo.read(StringIO(newick), "newick")
for tip in tree.get_terminals():
    tip.name = taxa[int(tip.name) - 1]

plt.style.use(ROOT / "notebooks/ncb.mplstyle")

In [ ]:
names = {
    "1999-101_Gryllus_bimaculatus": "G. bimaculatus (BIM)",
    "G62_Gryllus_firmus_*Florida": "G. firmus (FIR)",
    "G30_Gryllus_veletis": "G. veletis (VEL)",
    "2016-034_Gryllus_staccato": "G. staccato (STA)",
    "2016-039_Gryllus_longicercus": "G. longicercus (LON)",
    "2016-037_Gryllus_regularis": "G. regularis (REG)",
    "G1357_Gryllus_personatus": "G. personatus (PER)",
    "d437_Gryllus_rubens": "G. rubens (RUB)",
    "G3335_Gryllus_vocalis": "G. vocalis (VOC)",
    "2016-033_Gryllus_lineaticeps": "G. lineaticeps (LIN)",
    "G3374_Gryllus_armatus": "G. armatus (ARM)",
    "G3382_Gryllus_texensis": "G. texensis (TEX)",
    "G3422_Gryllus_saxatilis_*tulare": "G. saxatilis 'Tulare' (TUL)",
    "G2665_Gryllus_lightfooti": "G. lightfooti (LIG)",
}
for tip in list(tree.get_terminals()):
    if tip.name not in names:
        tree.prune(tip)
tree.ladderize(reverse=True)

In [ ]:
def tip_label(clade):
    if not clade.is_terminal():
        return None
    genus, species, rest = names[clade.name].split(" ", 2)
    return rf"$\mathit{{{genus}\ {species}}}$ {rest}"

fig, ax = plt.subplots(figsize=(7.5, 5.0))
Phylo.draw(tree, axes=ax, do_show=False, show_confidence=False,
           label_func=tip_label)
for label in ax.texts:
    label.set_fontsize(11)
ax.set_xlim(0, 0.017)
ax.set_xticks([0, 0.004, 0.008, 0.012, 0.016])
ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_xlabel("Substitutions per site")
ax.set_ylabel("")
ax.set_yticks([])
fig.tight_layout()
fig.savefig(FIGURES / "fig_S4.pdf", bbox_inches="tight")
plt.show()